In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install autogluon.tabular
!pip install -q pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os, random, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
import autogluon.tabular
print("✅ autogluon OK:", autogluon.tabular.__version__)

# ===============================================================
# 0️⃣ Config / Reproducibility
# ===============================================================
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 21
print(f"✅ Using device: {DEVICE}")

# 데이터 경로 설정
DATA_DIR = "/content/drive/MyDrive/26-1 머신러닝 팀플/data"
EXP_TAG = "v10_snn_seedens"   # ⭐ Round 1: SNN 3-seed ensemble
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models", EXP_TAG)
CACHE_DIR = os.path.join(DATA_DIR, "cache", EXP_TAG)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ===============================================================
# 1️⃣ Dataset Wrapper
# ===============================================================
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

✅ autogluon OK: 1.5.0
✅ Using device: cuda


In [ ]:
# ===============================================================
# 2️⃣ MLP Models (GELU / SNN)
# ===============================================================
class ResidualBlock(nn.Module):
    """Residual block with normalization and activation."""
    def __init__(self, dim, dropout=0.2, act="relu"):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
    def forward(self, x):
        residual = x
        out = self.act_fn(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        out = self.act_fn(out + residual)
        return out

class MLPEncoderWithHead(nn.Module):
    """Encoder + projection + classification head (GELU/ReLU)"""
    def __init__(self, input_dim, act="relu", num_classes=NUM_CLASSES):
        super().__init__()
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act),
            nn.Linear(256, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act)
        )
        self.embedding = nn.Linear(256, 128)
        self.projection = nn.Sequential(nn.Linear(128, 128), self.act_fn, nn.Linear(128, 128))
        self.classifier = nn.Sequential(nn.LayerNorm(128), self.act_fn, nn.Linear(128, num_classes))
    def forward(self, x):
        h = self.backbone(x)
        emb = self.embedding(h)
        z = self.projection(F.normalize(emb, dim=1))
        logits = self.classifier(emb)
        return emb, z, logits


# ===============================================================
# 🆕 SNN (Self-Normalizing Network) - ReLU MLP 대체용
# - SELU + AlphaDropout으로 self-normalization (BatchNorm 불필요)
# - 작은 데이터셋 + 균등 클래스 문제에 강함
# - GELU MLP와 정규화 메커니즘 자체가 달라서 진짜 다양성
# ===============================================================
class SNNEncoderWithHead(nn.Module):
    """Self-Normalizing Network (Klambauer et al. 2017)"""
    def __init__(self, input_dim, num_classes=NUM_CLASSES, dropout=0.1):
        super().__init__()
        # ⭐ BatchNorm 없음. SELU가 자체 정규화
        # ⭐ AlphaDropout이 SELU 분포 보존
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.SELU(),
            nn.AlphaDropout(dropout),
            nn.Linear(256, 256),
            nn.SELU(),
            nn.AlphaDropout(dropout),
            nn.Linear(256, 256),
            nn.SELU(),
            nn.AlphaDropout(dropout),
        )
        self.embedding = nn.Linear(256, 128)
        self.projection = nn.Sequential(
            nn.Linear(128, 128), nn.SELU(), nn.Linear(128, 128)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(128), nn.SELU(), nn.Linear(128, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        """SELU 전용 LeCun Normal 초기화"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.backbone(x)
        emb = self.embedding(h)
        z = self.projection(F.normalize(emb, dim=1))
        logits = self.classifier(emb)
        return emb, z, logits

In [ ]:
# ===============================================================
# 3️⃣ Loss Functions
# ===============================================================
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss"""
    def __init__(self, temperature=0.1):
        super().__init__(); self.temperature = temperature
    def forward(self, features, labels):
        device = features.device; batch_size = features.shape[0]
        labels = labels.contiguous().view(-1,1)
        mask = torch.eq(labels, labels.T).float().to(device)
        anchor_dot = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot, dim=1, keepdim=True)
        logits = anchor_dot - logits_max.detach()
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True)+1e-12)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1)+1e-12)
        loss = -mean_log_prob_pos
        return loss.mean()

class CenterLoss(nn.Module):
    """Center Loss: improves intra-class compactness"""
    def __init__(self, num_classes, feat_dim, lambda_c=1.0):
        super().__init__()
        self.register_buffer('centers', torch.randn(num_classes, feat_dim))
        self.lambda_c = lambda_c
    def forward(self, emb, labels):
        centers_batch = self.centers[labels]
        return self.lambda_c * ((emb - centers_batch)**2).sum() / (2.0 * emb.size(0))
    @torch.no_grad()
    def update_centers(self, emb, labels, lr=0.5):
        for c in torch.unique(labels):
            mask = labels==c
            if mask.sum()==0:
                continue
            mean = emb[mask].mean(dim=0)
            self.centers[c] = (1-lr)*self.centers[c] + lr*mean

In [ ]:
# ===============================================================
# 🆕 LDA Feature Augmentation Helper (fold-safe)
# - Mahalanobis 제거 (소거법으로 망친 요소 확인됨)
# - LDA만 사용
# ===============================================================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

class LDAFeatureAugmenter:
    """
    Fold-safe LDA projection augmenter.
    - fit() : train fold에만 LDA + post-scaler fit
    - transform() : val/test에 transform 후 concat

    Returns concatenated [raw_scaled | lda_projection_scaled]
    """
    def __init__(self, n_components=None, shrinkage='auto'):
        self.n_components = n_components
        self.shrinkage = shrinkage
        self.lda = None
        self.post_scaler = None

    def fit(self, X_scaled, y, num_classes):
        max_comp = min(X_scaled.shape[1], num_classes - 1)
        n_comp = max_comp if self.n_components is None else min(self.n_components, max_comp)

        self.lda = LinearDiscriminantAnalysis(
            n_components=n_comp,
            solver='eigen',
            shrinkage=self.shrinkage
        )
        self.lda.fit(X_scaled, y)

        lda_proj = self.lda.transform(X_scaled)
        self.post_scaler = StandardScaler().fit(lda_proj)
        return self

    def transform(self, X_scaled):
        lda_proj = self.lda.transform(X_scaled)
        lda_proj = self.post_scaler.transform(lda_proj)
        return np.hstack([X_scaled, lda_proj])

    @property
    def out_dim_info(self):
        return f"raw={self.lda.n_features_in_}, lda={self.lda.n_components}"


def diagnose_lda(lda_aug, X_val_scaled, y_val):
    """LDA가 실제로 분리력 있는지 빠르게 확인 (logistic regression sanity check)"""
    from sklearn.linear_model import LogisticRegression
    proj = lda_aug.lda.transform(X_val_scaled)
    lr = LogisticRegression(max_iter=200, n_jobs=-1).fit(proj, y_val)
    return lr.score(proj, y_val)

In [ ]:
# ===============================================================
# 4️⃣ MLP Training (GELU / SNN 분기)
# ===============================================================
def evaluate_model(model, loader):
    """Evaluate model accuracy & F1"""
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, _, logits = model(xb)
            preds.append(torch.softmax(logits, dim=1).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds, trues = np.vstack(preds), np.concatenate(trues)
    acc = accuracy_score(trues, preds.argmax(1))
    f1 = f1_score(trues, preds.argmax(1), average="macro")
    return acc, f1, preds

def train_mlp_model(X, y, X_test, act="relu", epochs=100, use_lda=True):
    """
    act 옵션:
    - "relu" / "gelu" : 기존 MLPEncoderWithHead
    - "snn"           : 🆕 SNN (SELU + AlphaDropout, BatchNorm 없음)
    """
    from sklearn.utils.class_weight import compute_class_weight
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds  = np.zeros((len(X), NUM_CLASSES))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 Fold {fold} | act={act} | use_lda={use_lda}")

        # === fold-safe scaling ===
        fold_scaler = StandardScaler()
        X_tr_s  = fold_scaler.fit_transform(X[tr_idx])
        X_val_s = fold_scaler.transform(X[val_idx])
        X_te_s  = fold_scaler.transform(X_test)

        # === LDA augmentation ===
        if use_lda:
            lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
            lda_aug.fit(X_tr_s, y[tr_idx], num_classes=NUM_CLASSES)
            X_tr_s  = lda_aug.transform(X_tr_s)
            X_val_s = lda_aug.transform(X_val_s)
            X_te_s  = lda_aug.transform(X_te_s)

            if fold == 0:
                X_val_raw_only = X_val_s[:, :X[tr_idx].shape[1]]
                sanity = diagnose_lda(lda_aug, X_val_raw_only, y[val_idx])
                print(f"  📐 LDA dims: {lda_aug.out_dim_info} | sanity LR acc: {sanity:.4f}")

        train_loader = DataLoader(TabularDataset(X_tr_s,  y[tr_idx]), batch_size=256, shuffle=True, drop_last=True)
        val_loader   = DataLoader(TabularDataset(X_val_s, y[val_idx]), batch_size=512, shuffle=False)
        test_loader  = DataLoader(TabularDataset(X_te_s), batch_size=512, shuffle=False)

        # 모델 선택 분기
        if act == "snn":
            model = SNNEncoderWithHead(input_dim=X_tr_s.shape[1], dropout=0.1).to(DEVICE)
            # SNN은 lr 약간 낮춤 (안정성)
            lr = 5e-4
        else:
            model = MLPEncoderWithHead(input_dim=X_tr_s.shape[1], act=act).to(DEVICE)
            lr = 1e-3

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

        cw = compute_class_weight('balanced',
                                  classes=np.arange(NUM_CLASSES),
                                  y=y[tr_idx])
        cw_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
        ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05, weight=cw_t)

        supcon = SupConLoss(0.1)
        center = CenterLoss(NUM_CLASSES, 128, 1.0).to(DEVICE)

        best_f1, best_state = -1, None
        patience = 20
        no_improve = 0
        for ep in range(epochs):
            model.train(); losses=[]
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                emb, z, logits = model(xb)
                loss = 0.5*supcon(z,yb) + 1.0*ce_loss(logits,yb) + 0.1*center(emb,yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                center.update_centers(emb.detach(), yb.detach())
                losses.append(loss.item())
            acc, f1, _ = evaluate_model(model, val_loader)
            scheduler.step(1-f1)
            print(f"Epoch {ep}: loss={np.mean(losses):.4f} val_f1={f1:.4f}")
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"⏹️ Early stop at epoch {ep}, best_f1={best_f1:.4f}")
                    break

        model.load_state_dict(best_state)
        torch.save(model.state_dict(), f"{MODEL_DIR}/mlp_{act}_fold{fold}.pth")

        _, _, val_proba = evaluate_model(model, val_loader)
        oof_preds[val_idx] = val_proba

        model.eval()
        all_test=[]
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(DEVICE)
                _,_,logits=model(xb)
                all_test.append(torch.softmax(logits,dim=1).cpu().numpy())
        test_preds += np.vstack(all_test)/skf.n_splits
    return test_preds, oof_preds

In [ ]:
# ===============================================================
# 🆕 Round 1: SNN Seed Ensemble Wrapper
# - train_mlp_model을 3개 시드로 돌려서 test/oof 평균
# - 시드는 fold-split도 함께 바꿔서 분산 최대화
# - 단일 SNN 대비 분산만 줄이는 안전한 변경
# ===============================================================
import copy

def train_snn_seed_ensemble(X, y, X_test, seeds=(42, 1337, 2024), epochs=100):
    """
    SNN을 여러 시드로 학습하고 test/oof 평균 반환.
    train_mlp_model은 그대로 사용, SEED만 동적으로 교체.
    """
    global SEED  # train_mlp_model 안에서 SEED를 참조하므로
    original_seed = SEED

    test_acc = np.zeros((len(X_test), NUM_CLASSES))
    oof_acc  = np.zeros((len(X), NUM_CLASSES))

    for i, s in enumerate(seeds):
        print(f"\n{'='*60}")
        print(f"🌱 SNN Seed Run {i+1}/{len(seeds)} | seed={s}")
        print(f"{'='*60}")

        # 시드 전역 세팅 (numpy / torch / random / SKF 모두 영향)
        SEED = s
        np.random.seed(s)
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
        import random
        random.seed(s)

        test_s, oof_s = train_mlp_model(X, y, X_test, act="snn", epochs=epochs)

        test_acc += test_s / len(seeds)
        oof_acc  += oof_s  / len(seeds)

        # 시드별 OOF F1 확인 (다양성 진단용)
        f1_s = f1_score(y, oof_s.argmax(1), average='macro')
        print(f"  ✅ Seed {s} OOF F1: {f1_s:.4f}")

    SEED = original_seed  # 원복

    # 최종 앙상블 F1
    f1_ens = f1_score(y, oof_acc.argmax(1), average='macro')
    print(f"\n🎯 SNN Seed-Ensemble OOF F1: {f1_ens:.4f}")
    print(f"   (단일 시드 평균 대비 ↑여야 정상)")

    return test_acc, oof_acc

In [ ]:
# ===============================================================
# 5️⃣ TabNet
# ===============================================================
from pytorch_tabnet.pretraining import TabNetPretrainer

def train_tabnet_kfold(X, y, X_test, n_splits=5, use_pretrain=True, use_lda=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds  = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    # ⭐ Pretraining은 LDA 없이 raw로 (test 데이터 활용이 목적)
    pretrainer = None
    if use_pretrain:
        print("🧪 TabNet self-supervised pretraining (raw features)...")
        # StandardScaler 한 번 적용 (전체 데이터 — pretraining은 unsupervised라 OK)
        # 하지만 정합성 위해 fold 외부 scaler를 별도로 운영
        pre_scaler = StandardScaler().fit(X)
        X_unlab = np.vstack([pre_scaler.transform(X), pre_scaler.transform(X_test)])
        pretrainer = TabNetPretrainer(
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-2),
            mask_type='entmax',
            n_d=32, n_a=32, n_steps=4, gamma=1.5,
            seed=SEED, verbose=0,
        )
        pretrainer.fit(
            X_train=X_unlab,
            max_epochs=100, patience=20,
            batch_size=2048, virtual_batch_size=256,
            pretraining_ratio=0.8,
        )

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🌲 TabNet Fold {fold+1}/{n_splits} | use_lda={use_lda}")

        # ⭐ fold-safe scaling (기존 TabNet 코드엔 없었지만 LDA 위해 필요)
        fold_scaler = StandardScaler()
        X_tr  = fold_scaler.fit_transform(X[tr_idx])
        X_val = fold_scaler.transform(X[val_idx])
        X_te  = fold_scaler.transform(X_test)

        if use_lda:
            lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
            lda_aug.fit(X_tr, y[tr_idx], num_classes=NUM_CLASSES)
            X_tr  = lda_aug.transform(X_tr)
            X_val = lda_aug.transform(X_val)
            X_te  = lda_aug.transform(X_te)

        y_tr, y_val = y[tr_idx], y[val_idx]

        # pretrainer dimension mismatch 주의
        # pretrainer는 raw dim으로 학습됨. LDA augmented input은 dim이 다름!
        # 해결책: LDA 쓸 때는 from_unsupervised=None으로 (pretrain 포기)
        # 또는: pretrain도 LDA augmented input으로 다시 학습 (더 좋지만 비쌈)
        use_pretrainer_this_fold = pretrainer if not use_lda else None

        model = TabNetClassifier(
            n_d=32, n_a=32, n_steps=4, gamma=1.5,
            lambda_sparse=1e-4,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-2),
            scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
            scheduler_params=dict(T_0=10, T_mult=1, eta_min=1e-5),
            mask_type='entmax',
            seed=SEED + fold,
            verbose=10,
        )
        model.fit(
            X_train=X_tr, y_train=y_tr,
            eval_set=[(X_val, y_val)],
            eval_name=['val'],
            eval_metric=['accuracy', 'logloss'],
            max_epochs=300, patience=40,
            batch_size=1024, virtual_batch_size=128,
            num_workers=2, drop_last=False,
            from_unsupervised=use_pretrainer_this_fold,  # ⭐ LDA면 None
            weights=1,
        )
        # ... 이하 동일 ...

        # 저장
        model.save_model(os.path.join(MODEL_DIR, f"tabnet_fold{fold}"))

        # 예측
        val_proba = model.predict_proba(X_val)
        oof_preds[val_idx] = val_proba
        test_preds += model.predict_proba(X_te) / n_splits

        f1 = f1_score(y_val, val_proba.argmax(1), average='macro')
        fold_scores.append(f1)
        print(f"  Fold {fold+1} macro-F1: {f1:.4f}")

    print(f"\n📊 TabNet CV mean F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
    return test_preds, oof_preds

In [ ]:
# ===============================================================
# 6️⃣ AutoGluon
# ===============================================================
def train_autogluon(train_df, test_df, use_lda=True):
    from autogluon.tabular import TabularPredictor
    label = 'target'

    # ⭐ ID 컬럼 방어 (있으면 제거)
    train_df = train_df.drop(columns=[c for c in ['ID'] if c in train_df.columns]).copy()
    test_df  = test_df.drop(columns=[c for c in ['ID'] if c in test_df.columns]).copy()

    if use_lda:
        # ⭐ AutoGluon은 내부 CV를 하므로, 여기선 전체 train에 LDA fit
        feature_cols = [c for c in train_df.columns if c != label]
        X_tr = train_df[feature_cols].values
        y_tr = train_df[label].values
        X_te = test_df[feature_cols].values

        scaler = StandardScaler().fit(X_tr)
        X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

        lda_aug = LDAFeatureAugmenter(n_components=20, shrinkage='auto')
        lda_aug.fit(X_tr_s, y_tr, num_classes=NUM_CLASSES)
        lda_tr = lda_aug.lda.transform(X_tr_s)
        lda_tr = lda_aug.post_scaler.transform(lda_tr)
        lda_te = lda_aug.lda.transform(X_te_s)
        lda_te = lda_aug.post_scaler.transform(lda_te)

        lda_cols = [f"lda_{i}" for i in range(lda_tr.shape[1])]
        train_df = pd.concat([
            train_df.reset_index(drop=True),
            pd.DataFrame(lda_tr, columns=lda_cols)
        ], axis=1)
        test_df = pd.concat([
            test_df.reset_index(drop=True),
            pd.DataFrame(lda_te, columns=lda_cols)
        ], axis=1)

    predictor = TabularPredictor(
        label=label,
        eval_metric='f1_macro',
        path=os.path.join(MODEL_DIR, "autogluon")
    ).fit(
        train_data=train_df,
        time_limit=3600,
        presets='best_quality'
    )
    return predictor.predict_proba(test_df).values

In [ ]:
# ===============================================================
# 7️⃣ Hungarian / Greedy Assignment
# ===============================================================
def compute_quota(probs):
    N,C=probs.shape; base=N//C; rem=N%C
    quota=np.full(C,base,int)
    strength=probs.mean(0); order=np.argsort(-strength)
    quota[order[:rem]]+=1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps=1e-15; N,C=probs.shape
    cols=[]; col_to_class=[]
    for c in range(C):
        q=int(quota[c])
        cost_c=-np.log(probs[:,c]+eps)
        cols.append(np.repeat(cost_c[:,None],q,axis=1))
        col_to_class.extend([c]*q)
    cost=np.concatenate(cols,axis=1)
    row_ind,col_ind=linear_sum_assignment(cost)
    pred=np.zeros(N,int); pred[row_ind]=np.array(col_to_class)[col_ind]
    return pred

def balanced_assign_greedy(probs, quota):
    N,C=probs.shape; remaining=quota.copy(); pred=-np.ones(N,int)
    order=np.argsort(-probs.max(1)); ranked=np.argsort(-probs,1)
    for idx in order:
        for c in ranked[idx]:
            if remaining[c]>0:
                pred[idx]=c; remaining[c]-=1; break
    if (pred<0).any():
        for idx in np.where(pred<0)[0]:
            c=int(np.argmax(remaining)); pred[idx]=c; remaining[c]-=1
    return pred


from scipy.optimize import minimize

def find_best_weights(oof_list, y_true):
    """OOF로 최적 ensemble weight 탐색 (macro F1 최대화)"""
    def neg_f1(w):
        w = np.abs(w); w = w / w.sum()
        blend = sum(wi * oof for wi, oof in zip(w, oof_list))
        return -f1_score(y_true, blend.argmax(1), average='macro')
    init = np.ones(len(oof_list)) / len(oof_list)
    res = minimize(neg_f1, init, method='Nelder-Mead',
                   options={'xatol':1e-4, 'maxiter':500})
    w = np.abs(res.x); w = w / w.sum()
    return w, -res.fun

In [ ]:
##Run all 들어가기 전 캐시 저장
# ===============================================================
# 🔖 Cache helper (Run all 들어가기 전 캐시 저장/로드)
# ===============================================================
CACHE_DIR = os.path.join(DATA_DIR, "cache", EXP_TAG)   # ⭐ 태그 추가
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_or_run(name, fn, *args, force=False, **kwargs):
    path = os.path.join(CACHE_DIR, f"{name}.npz")
    if os.path.exists(path) and not force:
        print(f"📦 Load cache: {path}")
        data = np.load(path, allow_pickle=True)
        return tuple(data[k] for k in data.files)
    print(f"🛠️  Run & cache: {name}")
    out = fn(*args, **kwargs)
    if not isinstance(out, tuple):
        out = (out,)
    np.savez(path, **{f"arr_{i}": a for i, a in enumerate(out)})
    return out

In [ ]:
def run_all():
    print("✅ Data files OK")
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    print(f"✅ FE skip | train: {train.shape}, test: {test.shape}")

    X = train.drop(columns=["target", "ID"]).values
    y = train["target"].values
    X_test = test.drop(columns=["ID"]).values

    print("\n🚀 Training TabNet (5-Fold)...")
    tab_test, tab_oof = cache_or_run(
        "tabnet", train_tabnet_kfold, X, y, X_test
    )

    # ⭐ Round 1: SNN seed ensemble (3 seeds)
    print("\n🚀 Training SNN Seed Ensemble (3 seeds)...")
    snn_test, snn_oof = cache_or_run(
        "mlp_snn_seedens", train_snn_seed_ensemble, X, y, X_test,
        seeds=(42, 1337, 2024)
    )

    print("\n🚀 Training MLP (GELU)...")
    gelu_test, gelu_oof = cache_or_run(
        "mlp_gelu", train_mlp_model, X, y, X_test, act="gelu"
    )

    print("\n🚀 Training AutoGluon...")
    (autogluon_test,) = cache_or_run(
        "autogluon", train_autogluon, train, test
    )

    # Auto-weight: 3-model (TabNet + SNN + GELU)
    print("\n⚖️ Auto-weight ensemble (3 models)")
    w, best = find_best_weights([tab_oof, snn_oof, gelu_oof], y)
    print(f"🎯 Weights: TabNet={w[0]:.3f} SNN={w[1]:.3f} GELU={w[2]:.3f} | OOF F1={best:.4f}")
    blend_3 = w[0]*tab_test + w[1]*snn_test + w[2]*gelu_test
    ensemble = 0.8*blend_3 + 0.2*autogluon_test

    # Probability 저장
    np.save(os.path.join(DATA_DIR, f'old_ensemble_proba_{EXP_TAG}.npy'), ensemble)
    np.save(os.path.join(DATA_DIR, f'old_blend3_proba_{EXP_TAG}.npy'), blend_3)
    np.save(os.path.join(DATA_DIR, f'old_autogluon_proba_{EXP_TAG}.npy'), autogluon_test)
    np.save(os.path.join(DATA_DIR, f'old_tab_oof_{EXP_TAG}.npy'), tab_oof)
    np.save(os.path.join(DATA_DIR, f'old_snn_oof_{EXP_TAG}.npy'), snn_oof)
    np.save(os.path.join(DATA_DIR, f'old_gelu_oof_{EXP_TAG}.npy'), gelu_oof)
    print(f"✅ Probabilities saved")

    # Hungarian assignment
    quota = compute_quota(ensemble)
    try:
        preds = balanced_assign_hungarian(ensemble, quota)
        method = "Hungarian"
    except Exception:
        preds = balanced_assign_greedy(ensemble, quota)
        method = "Greedy"

    submission = pd.DataFrame({"ID": test["ID"], "target": preds})
    output_path = os.path.join(DATA_DIR, f"submission_final_ensemble_{EXP_TAG}.csv")
    submission.to_csv(output_path, index=False)
    print(f"\n✅ Final saved: {output_path} ({method} used)")

In [ ]:
run_all()

✅ Data files OK
✅ FE skip | train: (21693, 54), test: (15004, 53)

🚀 Training TabNet (5-Fold)...
📦 Load cache: /content/drive/MyDrive/26-1 머신러닝 팀플/data/cache/v10_snn_seedens/tabnet.npz

🚀 Training SNN Seed Ensemble (3 seeds)...
📦 Load cache: /content/drive/MyDrive/26-1 머신러닝 팀플/data/cache/v10_snn_seedens/mlp_snn_seedens.npz

🚀 Training MLP (GELU)...
📦 Load cache: /content/drive/MyDrive/26-1 머신러닝 팀플/data/cache/v10_snn_seedens/mlp_gelu.npz

🚀 Training AutoGluon...
🛠️  Run & cache: autogluon


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          12
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 79.25/79.25 GB
Total GPU Memory:   Free: 79.25 GB, Allocated: 0.00 GB, Total: 79.25 GB
GPU Count:          1
Memory Avail:       163.67 GB / 167.05 GB (98.0%)
Disk Space Avail:   9.51 GB / 100.00 GB (9.5%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic

[1000]	valid_set's multi_logloss: 0.713219	valid_set's f1_macro: 0.78294


	0.7945	 = Validation score   (f1_macro)
	230.77s	 = Training   runtime
	3.2s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 134.88s of the 434.94s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	Ran out of time, early stopping on iteration 177. Best iteration is:
	[156]	valid_set's multi_logloss: 0.514651	valid_set's f1_macro: 0.809339
	Ran out of time, early stopping on iteration 184. Best iteration is:
	[144]	valid_set's multi_logloss: 0.529105	valid_set's f1_macro: 0.814319
	Ran out of time, early stopping on iteration 193. Best iteration is:
	[123]	valid_set's multi_logloss: 0.520622	valid_set's f1_macro: 0.80556
	Ran out of time, early stopping on iteration 205. Best iteration is:
	[101]	valid_set's multi_logloss: 0.553303	valid_set's f1_macro: 0.800436
	Ran out of time, early stopping on iteration 216. Best iteration is:
	[201]	valid_set's multi_logloss: 0.